# Rol 4 — El Estratega: dinámica poblacional del AG
Ciclo completo (`src/rol4_ag.py`) y experimentos (`src/rol4_experimentos.py`).
Se estudia **tamaño de población, elitismo, número de generaciones y mutación**, y se mide la convergencia
(máx, media, desviación, diversidad, evaluaciones). Los operadores son los del Rol 3 (combinación C3).

In [1]:
import sys, os, importlib
sys.path.insert(0, os.path.abspath(".."))
os.chdir(os.path.abspath(".."))          # raíz del repo
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from src import rol3_operadores as op, rol4_ag as rol4, rol4_experimentos as e4, rol5_analisis as an

def _cargar(nombre):
    try:    return importlib.import_module(f"src.{nombre}"), True
    except ModuleNotFoundError: return importlib.import_module("src._provisional"), False
rol1, ok1 = _cargar("rol1_codificacion"); rol2, ok2 = _cargar("rol2_fitness")
if not (ok1 and ok2):
    print("⚠️  FITNESS/CODIFICACIÓN PROVISIONALES (Roles 1 y 2 aún no subidos). Repetir con Restart & Run All cuando estén.")
GRAF, TAB = "results/graficos", "results/tablas"
os.makedirs(GRAF, exist_ok=True); os.makedirs(TAB, exist_ok=True)
SEMILLAS = range(5)
base = {"poblacion": 30, "n_genes": rol1.N_GENES, "generaciones": 50, "elitismo": 2,
        "operadores": op.COMBINACIONES["C3 Balance (torneo + blend α=0.5 + no-uniforme)"]}
print("presupuesto base =", base["poblacion"] + base["generaciones"]*(base["poblacion"]-base["elitismo"]), "evaluaciones")

⚠️  FITNESS/CODIFICACIÓN PROVISIONALES (Roles 1 y 2 aún no subidos). Repetir con Restart & Run All cuando estén.
presupuesto base = 1430 evaluaciones


## 1. Ciclo del AG (resumen)
Por generación: ordenar → **elites** pasan intactos → `reproducir` crea `poblacion − elitismo` hijos →
se evalúan solo los hijos → se registra max/media/std/diversidad/evaluaciones → ¿parada?
Evaluaciones totales = `N + T·(N − elitismo)`.

## 2. Configuraciones (cada una cambia un solo parámetro) y corridas

In [2]:
cfgs = e4.configuraciones(base)
df, hist, mejores = e4.correr_configuraciones(rol4.ejecutar_ag, rol2.fitness, cfgs, seeds=SEMILLAS)
tabla = e4.tabla_resumen(df)
tabla.to_csv(f"{TAB}/rol4_configuraciones.csv"); df.to_csv(f"{TAB}/rol4_corridas.csv", index=False)
tabla

,poblacion,elitismo,generaciones,evaluaciones,fit_final,desv,fit_400ev,div_final,gen_colapso,% prematura,Δ vs base
config,,,,,,,,,,,
base,30,2,50.0,1430.0,0.9968,0.0023,0.9894,0.0199,40.4000,0.0,0.0000
pob=10,10,2,50.0,410.0,0.9927,0.0028,0.9927,0.0017,27.6000,0.0,-0.0041
pob=20,20,2,50.0,920.0,0.9955,0.0021,0.9911,0.0097,41.0000,0.0,-0.0014
pob=60,60,2,50.0,2960.0,0.9984,0.0012,0.9845,0.0405,42.6667,0.0,0.0016
elit=0,30,0,50.0,1530.0,0.9976,0.0011,0.9887,0.0258,43.5000,0.0,0.0008
elit=1,30,1,50.0,1480.0,0.9984,0.0006,0.9893,0.0207,42.0000,0.0,0.0016
elit=5,30,5,50.0,1280.0,0.9978,0.0011,0.9918,0.0220,39.2500,0.0,0.0010
elit=10,30,10,50.0,1030.0,0.9968,0.0005,0.9915,0.0178,42.4000,0.0,-0.0001
gen=25,30,2,25.0,730.0,0.9941,0.0015,0.9890,0.0644,24.5000,0.0,-0.0028


**Cómo leer la tabla:** `fit_final` = mejor fitness medio de 5 semillas; `fit_400ev` = mejor fitness usando
solo ~400 evaluaciones (mismo costo para todos); `div_final` = diversidad al terminar; `gen_colapso` = primera
generación con diversidad < 0.05; `% prematura` = corridas con diversidad < 0.05 y fitness lejos del mejor conocido.

## 3. Curvas de convergencia

In [3]:
e4.graficar_convergencia(hist, ["base", "pob=10", "pob=60"], GRAF, "rol4_conv_poblacion.png")
e4.graficar_convergencia(hist, ["elit=0", "elit=2", "elit=10"] if "elit=2" in hist else ["elit=0", "base", "elit=10"], GRAF, "rol4_conv_elitismo.png")
e4.graficar_convergencia(hist, ["gen=25", "base", "gen=100"], GRAF, "rol4_conv_generaciones.png")
e4.graficar_vs_evaluaciones(hist, ["pob=10", "base", "pob=60"], GRAF, "rol4_vs_evaluaciones.png")
plt.show()

## 4. ¿Qué hiperparámetro importa más? (dos lecturas)
Cambiar la población cambia también el **costo**. Por eso se mira (a) con las mismas generaciones y
(b) con el mismo presupuesto de evaluaciones.

In [4]:
a, b, c = e4.efecto_hiperparametros(df)
print("(a) mismas generaciones"); display(a)
print("(b) mismo presupuesto (~400 evaluaciones)"); display(b)
print("(c) población a IGUAL presupuesto total (10 vs 60, ~1430 evaluaciones)"); display(c)
an.graficar_tornado(a, "Rol 4 — efecto con mismas generaciones", GRAF, "rol4_tornado_generaciones.png")
an.graficar_tornado(b, "Rol 4 — efecto con mismo presupuesto (~400 ev.)", GRAF, "rol4_tornado_presupuesto.png")
plt.show()

(a) mismas generaciones


,factor,valor_bajo,valor_alto,fitness_bajo,fitness_alto,base,rango
0,poblacion,10,60,0.992750,0.998431,0.996845,0.005681
1,generaciones,25,100,0.994095,0.999390,0.996845,0.005295
2,elitismo,0,10,0.997596,0.996784,0.996845,0.000812
3,mutación pm,0.02,0.3,0.996177,0.996398,0.996845,0.000221


(b) mismo presupuesto (~400 evaluaciones)


,factor,valor_bajo,valor_alto,fitness_bajo,fitness_alto,base,rango
0,poblacion,10,60,0.992745,0.984543,0.989397,0.008202
1,elitismo,0,10,0.988745,0.991500,0.989397,0.002756
2,mutación pm,0.02,0.3,0.988912,0.989211,0.989397,0.000299


(c) población a IGUAL presupuesto total (10 vs 60, ~1430 evaluaciones)


,factor,valor_bajo,valor_alto,fitness_bajo,fitness_alto,base,rango
0,poblacion (=presup.),10,60,0.998502,0.995037,0.996845,0.003466


## 5. Convergencia prematura y deriva genética

In [5]:
print("Corridas con convergencia prematura:", int(df["prematura"].sum()), "de", len(df))
print("Configuraciones con diversidad final más baja:")
display(tabla["div_final"].sort_values().head(3).to_frame())
e4.graficar_estancamiento(hist, "pob=10", GRAF, "rol4_estancamiento_pob10.png"); plt.show()
nombres = [g[0] if isinstance(g, (tuple, list)) else str(g) for g in getattr(rol1, "GENES", range(rol1.N_GENES))]
deriva = e4.deriva_por_gen(mejores, nombres, "base"); deriva.to_csv(f"{TAB}/rol4_deriva_genes.csv", index=False)
deriva

Corridas con convergencia prematura: 0 de 80
Configuraciones con diversidad final más baja:


,div_final
config,
pob=10 (=presup.),0.0003
pob=10,0.0017
gen=100,0.0039


,gen,desv_entre_semillas,media
0,amp_h1,0.009,0.984
1,ataque,0.009,0.025
2,amp_h8,0.011,0.027
3,sostenido,0.024,0.172
4,amp_h6,0.030,0.107
5,amp_h7,0.039,0.074
6,amp_h5,0.056,0.156
7,decaimiento,0.064,0.352
8,vib_prof,0.070,0.037
9,amp_h4,0.087,0.319


**Lectura:** un gen con desviación entre semillas ≈ 0.29 (la de un uniforme en [0,1]) es casi neutral:
cada corrida lo fija en un valor al azar (deriva). Un gen con desviación baja es importante: todas las corridas
convergen al mismo valor.

In [6]:
ejemplo = rol4.ejecutar_ag({**base, "paciencia": 10, "generaciones": 200}, rol2.fitness, 0)
print("Parada:", ejemplo["motivo_parada"], "en la generación", ejemplo["gen_final"],
      "| evaluaciones:", ejemplo["historial"]["evaluaciones"][-1], "| mejor fitness:", round(ejemplo["mejor_fit"], 4))

Parada: estancamiento en la generación 98 | evaluaciones: 2774 | mejor fitness: 0.9991
